In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
import matplotlib.pyplot as plt

import os

os.environ["NUMEXPR_MAX_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

import numpy as np
from pathlib import Path

processed_path = Path("/Users/carlasequero/Desktop/trabajo-fin-master/solution/data/splits/processed_data.npz")

data = np.load(processed_path)

print(data.files)

In [9]:
# Datos de entrada Gaia
X_train = data["X_train"]
X_val = data["X_val"]
X_test = data["X_test"]

# Errores de Gaia
X_err_train = data["X_err_train"]
X_err_val = data["X_err_val"]
X_err_test = data["X_err_test"]

# IDs de Gaia
X_id_train = data["X_id_train"]
X_id_val = data["X_id_val"]
X_id_test = data["X_id_test"]

# Datos objetivo SDSS
y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

# IDs de SDSS
y_id_train = data["y_id_train"]
y_id_val = data["y_id_val"]
y_id_test = data["y_id_test"]

# Longitudes de onda
gaia_wavelength = data["gaia_wavelength"]
sdss_wavelength = data["sdss_wavelength"]

In [10]:
X_scale_train = np.nanmedian(np.abs(X_train), axis=1, keepdims=True)
X_scale_val = np.nanmedian(np.abs(X_val), axis=1, keepdims=True)
X_scale_test = np.nanmedian(np.abs(X_test), axis=1, keepdims=True)

y_scale_train = np.nanmedian(np.abs(y_train), axis=1, keepdims=True)
y_scale_val = np.nanmedian(np.abs(y_val), axis=1, keepdims=True)
y_scale_test = np.nanmedian(np.abs(y_test), axis=1, keepdims=True)

X_train_norm_median_spec = X_train / X_scale_train
X_val_norm_median_spec = X_val / X_scale_val
X_test_norm_median_spec = X_test / X_scale_test

y_train_norm_median_spec = y_train / y_scale_train
y_val_norm_median_spec = y_val / y_scale_val
y_test_norm_median_spec = y_test / y_scale_test

In [11]:
X_train_rnn = X_train_norm_median_spec[..., np.newaxis]
X_val_rnn = X_val_norm_median_spec[..., np.newaxis]
X_test_rnn = X_test_norm_median_spec[..., np.newaxis]

print("X_train_rnn:", X_train_rnn.shape)
print("X_val_rnn:", X_val_rnn.shape)
print("X_test_rnn:", X_test_rnn.shape)

In [ ]:
n_gaia = X_train_rnn.shape[1]
n_features = X_train_rnn.shape[2]
n_sdss = y_train_norm_median_spec.shape[1]

model_rnn = models.Sequential([
    layers.Input(shape=(n_gaia, n_features)),

    layers.GRU(128, return_sequences=True),

    layers.GRU(128, return_sequences=False ),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(n_sdss)
])

model_rnn.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae", "mse"]
)

model_rnn.summary()

In [13]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    min_delta=1e-4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [14]:
history_rnn = model_rnn.fit(
    X_train_rnn,
    y_train_norm_median_spec,
    validation_data=(X_val_rnn, y_val_norm_median_spec),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [15]:
test_results_rnn = model_rnn.evaluate(
    X_test_rnn,
    y_test_norm_median_spec,
    verbose=1
)

In [16]:
model_rnn_small = models.Sequential([
    layers.Input(shape=(n_gaia, n_features)),

    layers.GRU(128, return_sequences=False),

    layers.Dense(512),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(1024),
    layers.LeakyReLU(negative_slope=0.01),

    layers.Dense(n_sdss)
])

model_rnn_small.compile(
    optimizer=Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=tf.keras.losses.Huber(delta=1.0),
    metrics=["mae", "mse"]
)

model_rnn_small.summary()

In [17]:
history_rnn_small = model_rnn_small.fit(
    X_train_rnn,
    y_train_norm_median_spec,
    validation_data=(X_val_rnn, y_val_norm_median_spec),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
test_results_small_rnn = model_rnn_small.evaluate(
    X_test_rnn,
    y_test_norm_median_spec,
    verbose=1
)